# OpenAPI와 TS SDK

Pydantic과 FastAPI 선언이 OpenAPI 스키마가 되고 TypeScript 클라이언트 생성 입력이 되는 과정을 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 스키마 생성

서버 프로세스를 시작하지 않아도 앱 객체에서 같은 OpenAPI 문서를 얻을 수 있습니다.

In [2]:
import json
from agent_service.app import app
schema = app.openapi()
path = Path('openapi.json')
path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')
print(schema['openapi'], len(schema['paths']))
assert '/v1/chat' in schema['paths']

3.1.0 5


## 타입과 보안 정보 관찰

요청 스키마의 필수 필드와 응답 타입이 계약 문서에 포함됩니다.

In [3]:
chat = schema['paths']['/v1/chat']['post']
request_ref = chat['requestBody']['content']['application/json']['schema']['$ref']
response_ref = chat['responses']['200']['content']['application/json']['schema']['$ref']
print(request_ref, response_ref)
assert request_ref.endswith('/ChatRequest')
assert response_ref.endswith('/ChatResponse')

#/components/schemas/ChatRequest #/components/schemas/ChatResponse


## 반복 실험과 결과 해석

같은 앱에서 두 번 생성한 스키마를 정렬해 직렬화하면 같은 결과가 나옵니다.

In [4]:
first = json.dumps(app.openapi(), sort_keys=True)
second = json.dumps(app.openapi(), sort_keys=True)
print('동일:', first == second, '문자 수:', len(first))
assert first == second
print('SDK 생성 입력:', path.resolve())

동일: True 문자 수: 4885
SDK 생성 입력: [사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/openapi.json
